# 网站宣传册生成器

一种人工智能驱动的工具，可以从任何网站自动生成专业手册。此笔记本提供了一种将小册子生成器与 Jupyter 笔记本结合使用的交互式方式。

## 特点

- 🌐 **网站分析**：自动抓取和分析网站内容
- 🤖 **AI-Powered**：使用 OpenAI GPT-4o-mini 进行智能内容生成
- 📄 **专业输出**：生成 markdown 格式的小册子
- 🌍 **多语言支持**：使用人工智能将宣传册翻译成任何语言
- ⚡ **交互式**：在 Jupyter 笔记本中逐步运行
- 🎨 **漂亮的输出**：具有 HTML 样式的原生 Jupyter markdown 渲染

## 先决条件

- Python 3.8 或更高版本
- OpenAI API 密钥
- Jupyter笔记本环境

## 设置说明

1. **获取您的 OpenAI API 密钥**：
   - 访问 [OpenAI API 密钥](https://platform.openai.com/api-keys)
   - 创建新的API密钥

2. **设置环境变量**：
   - 在项目目录中创建一个“.env”文件：“OPENAI_API_KEY=your_api_key_here”
   - 或者直接在笔记本中设置环境变量

3. **安装依赖**：
   ```bash
   pip install openai python-dotenv requests beautifulsoup4 ipywidgets
   ```

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 导入所需的库
from openai import OpenAI
from dotenv import load_dotenv
import os
import requests
import json
from typing import List
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output
import time

print("✅ All libraries imported successfully!")



## 配置

设置 OpenAI API 密钥并配置客户端。

In [ ]:
# 配置单元 - 设置您的 OpenAI API 密钥
def get_client_and_headers():
    """Initialize OpenAI client and headers for web scraping"""
    load_dotenv(override=True)
    api_key = os.getenv("OPENAI_API_KEY")
    
    if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
        print("✅ API key looks good!")
    else:
        print("⚠️  There might be a problem with your API key")
        print("Make sure you have set OPENAI_API_KEY in your .env file or environment variables")

    client = OpenAI(api_key=api_key)
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    return client, headers

# 初始化客户端
client, headers = get_client_and_headers()
print("✅ OpenAI client initialized successfully!")



## 核心功能

主要功能为网站分析和宣传册生成。

In [ ]:
# 以 Markdown 格式显示内容的实用方法
def display_content(content, is_markdown=True):
    """Display content using Jupyter's display methods"""
    if is_markdown:
        display(Markdown(content))
    else:
        print(content)

def stream_content(response, title="Content"):
    """
    Utility function to handle streaming content display in Jupyter
    
    Args:
        response: OpenAI streaming response object
        title (str): Title to display for the streaming content
    
    Returns:
        str: Complete streamed content
    """
    result = ""
    
    # 显示标题
    display(HTML(f"<h3 style='color: #1f77b4;'>{title}...</h3>"))
    
    # 创建用于流式传输的输出小部件
    from IPython.display import clear_output
    import time
    
    for chunk in response:
        content = chunk.choices[0].delta.content or ""
        result += content
        # 打印每个块到达时的流效果
        print(content, end='', flush=True)
    
    # 显示完成消息
    display(HTML(f"<div style='color: green; font-weight: bold; margin-top: 20px;'>{'='*50}</div>"))
    display(HTML(f"<div style='color: green; font-weight: bold;'>{title.upper()} COMPLETE</div>"))
    display(HTML(f"<div style='color: green; font-weight: bold;'>{'='*50}</div>"))
    
    return result

print("✅ Utility functions loaded!")



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 获取网站内容的实用程序类
class Website:
    def __init__(self, url):
        self.url = url
        self.client, self.headers = get_client_and_headers()
        print(f"🌐 Fetching content from: {url}")
        response = requests.get(url, headers=self.headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]
        print(f"✅ Website analyzed: {self.title}")

    def get_contents(self):
        return f"Webpage Title: {self.title}\nWebpage Contents: {self.text}\n\n"

print("✅ Website class loaded!")



In [ ]:
# AI提示功能
def get_links_system_prompt():
    link_system_prompt = """"You are provided with a list of links found on a webpage. \
        You are able to decide which of the links would be most relevant to include in a brochure about the company. \
        Relevant links usually include: About page, or a Company page, or Careers/Jobs pages or News page\n"""
    link_system_prompt += "Always respond in JSON exactly like this: \n"
    link_system_prompt += """
        {
            "links": [
                {"type": "<page type>", "url": "<full URL>"},
                {"type": "<page type>", "url": "<full URL>"}
            ]
        }\n
    """
    link_system_prompt += """ If no relevant links are found, return:
        {
            "links": []
        }\n
    """
    link_system_prompt += "If multiple links could map to the same type (e.g. two About pages), include the best candidate only.\n"

    link_system_prompt += "You should respond in JSON as in the below examples:\n"
    link_system_prompt += """
        # 实施例1
        Input links:
        - https://acme.com/about  
        - https://acme.com/pricing  
        - https://acme.com/blog  
        - https://acme.com/signup  

        Output:
        {
        "links": [
            {"type": "about page", "url": "https://acme.com/about"},
            {"type": "blog page", "url": "https://acme.com/blog"},
            {"type": "pricing page", "url": "https://acme.com/pricing"}
        ]
        }
        """
    link_system_prompt += """
        # 实施例2
        Input links:
        - https://startup.io/  
        - https://startup.io/company  
        - https://startup.io/careers  
        - https://startup.io/support  

        Output:
        {
        "links": [
            {"type": "company page", "url": "https://startup.io/company"},
            {"type": "careers page", "url": "https://startup.io/careers"}
        ]
        }
        """
    link_system_prompt += """
        # 实施例3
        Input links:
        - https://coolapp.xyz/login  
        - https://coolapp.xyz/random  

        Output:
        {
        "links": []
        }
        """
    return link_system_prompt

def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \n"
    user_prompt += "Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

def get_brochure_system_prompt():
    brochure_system_prompt = """
        You are an assistant that analyzes the contents of several relevant pages from a company website \
        and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.
        Include details of company culture, customers and careers/jobs if you have the information.
    """
    return brochure_system_prompt

def get_brochure_user_prompt(url):
    user_prompt = f"You are looking at a company details of: {url}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_details_for_brochure(url)
    user_prompt = user_prompt[:15000] # Truncate if more than 15,000 characters
    return user_prompt

def get_translation_system_prompt(target_language):
    translation_system_prompt = f"You are a professional translator specializing in business and marketing content. \
    Translate the provided brochure to {target_language} while maintaining all formatting and professional tone."
    return translation_system_prompt

def get_translation_user_prompt(original_brochure, target_language):
    translation_prompt = f"""
    You are a professional translator. Please translate the following brochure content to {target_language}.
    
    Important guidelines:
    - Maintain the markdown formatting exactly as it appears
    - Keep all headers, bullet points, and structure intact
    - Translate the content naturally and professionally
    - Preserve any company names, product names, or proper nouns unless they have established translations
    - Maintain the professional tone and marketing style
    
    Brochure content to translate:
    {original_brochure}
    """
    return translation_prompt

print("✅ AI prompt functions loaded!")



In [ ]:
# 核心宣传册生成功能
def get_links(url):
    """Get relevant links from a website using AI analysis"""
    website = Website(url)
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": get_links_system_prompt()},
            {"role": "user", "content": get_links_user_prompt(website)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print("🔗 Found relevant links:", result)
    return json.loads(result)

def get_details_for_brochure(url):
    """Get comprehensive details from website and relevant pages"""
    website = Website(url)
    result = "Landing page:\n"
    result += website.get_contents()
    links = get_links(url)
    print("📄 Analyzing additional pages...")
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

def create_brochure(url):
    """Create a brochure from a website URL"""
    website = Website(url)
    print("🤖 Generating brochure with AI...")
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": get_brochure_system_prompt()},
            {"role": "user", "content": get_brochure_user_prompt(url)}
        ]
    )
    result = response.choices[0].message.content
    display_content(result, is_markdown=True)
    return result

def stream_brochure(url):
    """Create a brochure with streaming output"""
    website = Website(url)
    print("🤖 Generating brochure with streaming output...")
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": get_brochure_system_prompt()},
            {"role": "user", "content": get_brochure_user_prompt(url)}
        ],
        stream=True
    )
    
    # 使用可重用的流实用程序函数
    result = stream_content(response, "Generating brochure")
    return result

print("✅ Core brochure generation functions loaded!")



In [ ]:
# 翻译功能
def translate_brochure(url, target_language="Spanish", stream_mode=False):
    """
    Generate a brochure and translate it to the target language
    
    Args:
        url (str): The website URL to generate brochure from
        target_language (str): The target language for translation (default: "Spanish")
        stream_mode (bool): Whether to use streaming output (default: False)
    
    Returns:
        str: Translated brochure content
    """
    # 首先生成原始宣传册
    print(f"🌍 Generating brochure and translating to {target_language}...")
    original_brochure = create_brochure(url)
    
    # 获取翻译提示
    translation_system_prompt = get_translation_system_prompt(target_language)
    translation_user_prompt = get_translation_user_prompt(original_brochure, target_language)
    
    # 获取 OpenAI 客户端
    website = Website(url)
    
    if stream_mode:
        # 使用 OpenAI 和流式传输生成翻译
        response = website.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": translation_system_prompt},
                {"role": "user", "content": translation_user_prompt}
            ],
            stream=True
        )
        
        # 使用可重用的流实用程序函数
        translated_brochure = stream_content(response, f"Translating brochure to {target_language}")
    else:
        # 使用 OpenAI 生成具有完整输出的翻译
        response = website.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": translation_system_prompt},
                {"role": "user", "content": translation_user_prompt}
            ]
        )
        
        translated_brochure = response.choices[0].message.content
        
        # 显示翻译内容
        display_content(translated_brochure, is_markdown=True)
    
    return translated_brochure

print("✅ Translation functions loaded!")



## 互动示例

现在让我们尝试为一些示例网站生成小册子。您可以运行这些单元来查看小册子生成器的运行情况！

In [ ]:
# 示例 1：为示例网站生成宣传册
# 您可以将此 URL 更改为您要分析的任何网站

sample_url = "https://openai.com"  # Change this to any website you want to analyze

print(f"🚀 Generating brochure for: {sample_url}")
print("=" * 60)

# 生成手册
brochure = create_brochure(sample_url)



In [ ]:
# 示例 2：生成带有流输出的小册子
# 这显示了实时生成的手册

streaming_url = "https://anthropic.com"  # Change this to any website you want to analyze

print(f"🚀 Generating brochure with streaming for: {streaming_url}")
print("=" * 60)

# 通过流式传输生成小册子
streaming_brochure = stream_brochure(streaming_url)



In [ ]:
# 示例 3：生成并翻译手册
# 这将创建一个小册子，然后将其翻译成另一种语言

translation_url = "https://huggingface.co"  # Change this to any website you want to analyze
target_language = "Spanish"  # Change this to any language you want

print(f"🚀 Generating and translating brochure for: {translation_url}")
print(f"🌍 Target language: {target_language}")
print("=" * 60)

# 生成并翻译手册
translated_brochure = translate_brochure(translation_url, target_language, stream_mode=False)



## 交互式小部件界面

使用下面的小部件以交互方式为任何网站生成小册子！

In [ ]:
# 交互式小部件界面
import ipywidgets as widgets
from IPython.display import display, clear_output

# 创建小部件
url_input = widgets.Text(
    value='https://openai.com',
    placeholder='Enter website URL (e.g., https://example.com)',
    description='Website URL:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

language_dropdown = widgets.Dropdown(
    options=['English', 'Spanish', 'French', 'German', 'Chinese', 'Japanese', 'Portuguese', 'Italian'],
    value='English',
    description='Language:',
    style={'description_width': 'initial'}
)

stream_checkbox = widgets.Checkbox(
    value=False,
    description='Use streaming output',
    style={'description_width': 'initial'}
)

translate_checkbox = widgets.Checkbox(
    value=False,
    description='Translate brochure',
    style={'description_width': 'initial'}
)

generate_button = widgets.Button(
    description='Generate Brochure',
    button_style='success',
    icon='rocket'
)

output_area = widgets.Output()

def on_generate_clicked(b):
    with output_area:
        clear_output(wait=True)
        url = url_input.value.strip()
        
        if not url:
            print("❌ Please enter a valid URL")
            return
            
        if not url.startswith(('http://', 'https://')):
            url = 'https://' + url
            
        print(f"🚀 Generating brochure for: {url}")
        print("=" * 60)
        
        try:
            if translate_checkbox.value:
                # 生成并翻译
                result = translate_brochure(url, language_dropdown.value, stream_mode=stream_checkbox.value)
            else:
                # 仅生成
                if stream_checkbox.value:
                    result = stream_brochure(url)
                else:
                    result = create_brochure(url)
            
            print("\n✅ Brochure generation completed!")
            
        except Exception as e:
            print(f"❌ Error generating brochure: {str(e)}")
            print("Please check your API key and internet connection.")

generate_button.on_click(on_generate_clicked)

# 显示小部件
print("🎯 Interactive Brochure Generator")
print("Enter a website URL and click 'Generate Brochure' to create a professional brochure!")
print()

display(url_input)
display(widgets.HBox([language_dropdown, stream_checkbox, translate_checkbox]))
display(generate_button)
display(output_area)



## 高级使用示例

以下是一些高级示例，展示了使用小册子生成器的不同方法。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 高级示例1：分析多个网站并进行比较
websites_to_analyze = [
    "https://openai.com",
    "https://anthropic.com", 
    "https://huggingface.co"
]

print("🔍 Analyzing multiple websites...")
print("=" * 60)

brochures = {}
for url in websites_to_analyze:
    print(f"\n📊 Generating brochure for: {url}")
    try:
        brochure = create_brochure(url)
        brochures[url] = brochure
        print(f"✅ Successfully generated brochure for {url}")
    except Exception as e:
        print(f"❌ Failed to generate brochure for {url}: {str(e)}")
    
    print("-" * 40)

print(f"\n🎉 Generated {len(brochures)} brochures successfully!")



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 高级示例 2：生成多种语言的宣传册
target_website = "https://openai.com"  # Change this to any website
languages = ["Spanish", "French", "German", "Chinese"]

print(f"🌍 Generating brochures in multiple languages for: {target_website}")
print("=" * 60)

multilingual_brochures = {}
for language in languages:
    print(f"\n🔄 Translating to {language}...")
    try:
        translated_brochure = translate_brochure(target_website, language, stream_mode=False)
        multilingual_brochures[language] = translated_brochure
        print(f"✅ Successfully translated to {language}")
    except Exception as e:
        print(f"❌ Failed to translate to {language}: {str(e)}")
    
    print("-" * 40)

print(f"\n🎉 Generated brochures in {len(multilingual_brochures)} languages!")



## 自定义函数

为特定用例创建您自己的自定义函数。

In [ ]:
# 自定义功能：将手册保存到文件
def save_brochure_to_file(brochure_content, filename, url):
    """Save brochure content to a markdown file"""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"# Brochure for {url}\n\n")
            f.write(f"Generated on: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.write("---\n\n")
            f.write(brochure_content)
        print(f"✅ Brochure saved to: {filename}")
        return True
    except Exception as e:
        print(f"❌ Error saving brochure: {str(e)}")
        return False

# 自定义功能：通过自定义分析生成手册
def generate_custom_brochure(url, focus_areas=None):
    """Generate a brochure with focus on specific areas"""
    if focus_areas is None:
        focus_areas = ["company overview", "products", "culture", "careers"]
    
    website = Website(url)
    
    # 带有焦点区域的自定义系统提示
    custom_system_prompt = f"""
    You are an assistant that analyzes website content and creates a professional brochure.
    Focus specifically on these areas: {', '.join(focus_areas)}.
    Create a markdown brochure that emphasizes these aspects for prospective customers, investors and recruits.
    """
    
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": custom_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(url)}
        ]
    )
    
    result = response.choices[0].message.content
    display_content(result, is_markdown=True)
    return result

# 自定义功能：快速网站分析
def quick_website_analysis(url):
    """Perform a quick analysis of a website without generating full brochure"""
    website = Website(url)
    
    analysis = f"""
    # 快速网站分析：{url}
    
    **Title:** {website.title}
    **Total Links Found:** {len(website.links)}
    **Content Length:** {len(website.text)} characters
    
    # 示例内容（前 500 个字符）：
    {website.text[:500]}...
    
    # 所有链接：
    {chr(10).join(website.links[:10])}  # Show first 10 links
    """
    
    display_content(analysis, is_markdown=True)
    return analysis

print("✅ Custom functions loaded!")



## 自定义函数的使用示例

使用我们刚刚创建的自定义函数尝试这些示例。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 示例：快速网站分析
test_url = "https://openai.com"  # Change this to any website

print("🔍 Performing quick website analysis...")
print("=" * 50)

quick_analysis = quick_website_analysis(test_url)



In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 示例：生成具有特定重点的自定义手册
custom_url = "https://anthropic.com"  # Change this to any website
focus_areas = ["AI safety", "research", "products", "team"]  # Custom focus areas

print("🎯 Generating custom brochure with specific focus...")
print(f"Focus areas: {', '.join(focus_areas)}")
print("=" * 50)

custom_brochure = generate_custom_brochure(custom_url, focus_areas)



In [ ]:
# 示例：生成手册并保存到文件
save_url = "https://huggingface.co"  # Change this to any website

print("💾 Generating brochure and saving to file...")
print("=" * 50)

# 生成手册
brochure_content = create_brochure(save_url)

# 保存到文件
filename = f"brochure_{save_url.replace('https://', '').replace('/', '_')}.md"
save_success = save_brochure_to_file(brochure_content, filename, save_url)

if save_success:
    print(f"📁 You can find the saved brochure in: {filename}")
else:
    print("❌ Failed to save brochure to file")



## 故障排除和提示

### 常见问题和解决方案

1. **API密钥问题**
   - 确保您的 OpenAI API 密钥已在“.env”文件中设置
   - 验证您的API密钥有足够的积分
   - 检查密钥是否以“sk-proj-”开头

2. **网站抓取问题**
   - 某些网站可能会阻止自动请求
   - 如果一个网站失败，请尝试不同的网站
   - 该工具使用标准的 User-Agent 标头来避免基本阻塞

3. **内存问题**
   - 大型网站可能会消耗大量内存
   - 该工具将内容截断为 15,000 个字符来管理此内容

4. **速率限制**
   - OpenAI 对 API 调用有速率限制
   - 如果您达到限制，请等待几分钟后再重试

### 获得更好结果的技巧

1. **选择好的网站**
   - 具有清晰的“关于”、“产品”和“职业”页面的网站效果最佳
   - 避免主要是图像或需要 JavaScript 的网站

2. **对长内容使用流式传输**
   - 启用流媒体，通过长手册获得更好的用户体验
   - 流媒体实时显示进度

3. **自定义重点领域**
   - 使用自定义宣传册功能专注于特定方面
   - 这可以帮助生成更有针对性的内容

4. **保存您的工作**
   - 使用保存功能保存手册以供日后参考
   - 文件以 Markdown 格式保存，方便编辑

## 结论

该 Jupyter 笔记本为网站手册生成器提供了全面的界面。您可以：

- ✅ 从任何网站生成专业手册
- ✅ 将小册子翻译成多种语言
- ✅ 使用交互式小部件，轻松操作
- ✅ 将小册子保存到文件中以供以后使用
- ✅ 执行快速网站分析
- ✅ 创建具有特定重点领域的定制手册
- ✅ 生成带有流输出的小册子以获取实时反馈

### 后续步骤

1. **尝试交互式小部件**：使用上面的小部件界面为您喜爱的网站生成小册子
2. **使用不同的 URL 进行实验**：使用各种类型的网站测试该工具
3. **探索翻译功能**：生成不同语言的小册子
4. **保存您的工作**：使用保存功能保存您生成的小册子
5. **定制重点领域**：创建针对公司特定方面的宣传册

### 支持

对于问题和疑问：
- 检查上面的故障排除部分
- 验证您的 OpenAI API 密钥是否已正确配置
- 确保您有稳定的互联网连接
- 如果一个网站失败，请尝试不同的网站

快乐的小册子生成！ 🚀